# Circuit Simplification - Benchmark (wall time + peak memory)

This notebook measures wall time and peak RSS memory of the
`CircuitSimplificationTask` across circuits and `n_processes` values, in order
to calibrate the placeholder constants in
`app/services/resource_estimation/circuit_simplification.py`:

- `FILTER_WORK_UNITS_PER_CORE` / `FILTER_WORK_UNITS_PER_CORE_HOUR`
- `BASE_MEMORY_GB` / `PER_WORKER_GB`

**Scope:** only the `single_compartment` algorithm is benchmarked (the only
algorithm that runs the expensive per-cell filter computation).

**Caveat:** numbers collected here are laptop numbers. They must be re-checked
on the launch system before updating the constants.


## Imports

In [ ]:
import os
import platform
import shutil
import socket
import subprocess
import sys
import tempfile
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

import obi_one as obi
from obi_one.core.info import Info
from obi_one.db_sdk import db_sdk
from obi_one.scientific.blocks.simplification_algorithms import (
    SingleCompartmentAlgorithm,
)
from obi_one.scientific.from_id.circuit_from_id import CircuitFromID
from obi_one.scientific.library.circuit import Circuit
from obi_one.scientific.tasks.circuit_simplification import (
    CircuitSimplificationScanConfig,
    CircuitSimplificationTask,
)


## Optional: entitycore staging client

Only needed when `USE_STAGING = True` (to fetch circuits by entity ID).
The notebook runs fully offline with local circuits by default.


In [ ]:
USE_STAGING = False

db_client = None
if USE_STAGING:
    from entitysdk import Client, ProjectContext
    from obi_auth import get_token

    virtual_lab_id = obi.LAB_ID_STAGING_TEST
    project_id = obi.PROJECT_ID_STAGING_TEST
    token = get_token(environment="staging")
    project_context = ProjectContext(
        virtual_lab_id=virtual_lab_id, project_id=project_id
    )
    db_client = Client(
        api_url="https://staging.openbraininstitute.org/api/entitycore",
        project_context=project_context,
        token_manager=token,
    )
    print("Connected to entitycore staging.")
else:
    print("USE_STAGING is False - running offline with local circuits.")

output_root = Path("../../../../../../obi-output/circuit_simplification").resolve()
output_root.mkdir(parents=True, exist_ok=True)
print(f"Output root: {output_root}")


## PeakMemorySampler

`sonata_simplify` runs filter computation in a `spawn` multiprocessing Pool,
so the main-process RSS alone misses worker memory. This sampler polls every
0.1 s and tracks, at each instant, the main-process RSS plus the summed RSS of
all (recursive) child processes. `peak_rss_total_gb` is the peak of the
instantaneous sum - not the sum of separate peaks.


In [ ]:
class PeakMemorySampler:
    """Sample main + child process RSS in a background thread."""

    def __init__(self, interval_s: float = 0.1):
        self.interval_s = interval_s
        self._stop = threading.Event()
        self._thread = None
        self._peak_main_bytes = 0
        self._peak_children_bytes = 0
        self._peak_total_bytes = 0
        self._n_samples = 0
        self._wall_s = 0.0
        self._t0 = None

    def _sample_once(self):
        proc = psutil.Process()
        main_rss = proc.memory_info().rss
        children_rss = 0
        for child in proc.children(recursive=True):
            try:
                children_rss += child.memory_info().rss
            except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
                continue
        self._peak_main_bytes = max(self._peak_main_bytes, main_rss)
        self._peak_children_bytes = max(self._peak_children_bytes, children_rss)
        self._peak_total_bytes = max(self._peak_total_bytes, main_rss + children_rss)
        self._n_samples += 1

    def _run(self):
        while not self._stop.is_set():
            try:
                self._sample_once()
            except psutil.Error:
                pass
            self._stop.wait(self.interval_s)

    def __enter__(self):
        self._t0 = time.perf_counter()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._wall_s = time.perf_counter() - self._t0
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=5)
        try:
            self._sample_once()
        except psutil.Error:
            pass
        return False

    def results(self) -> dict:
        gb = 1 / 1024**3
        return {
            "wall_s": self._wall_s,
            "peak_rss_main_gb": self._peak_main_bytes * gb,
            "peak_rss_children_gb": self._peak_children_bytes * gb,
            "peak_rss_total_gb": self._peak_total_bytes * gb,
            "n_samples": self._n_samples,
        }


## Benchmark matrix

`BENCH_N_PROCESSES` / `BENCH_CIRCUITS` env vars (comma-separated) can restrict
the matrix for smoke runs. `MAX_FILTER_LOCATIONS` must match
`DEFAULT_MAX_FILTER_LOCATIONS` in the resource estimator.


In [ ]:
TINY = Path("../../../../data/tiny_circuits")

CIRCUITS = {
    "N_10__top_nodes_dim6": {
        "path": TINY / "N_10__top_nodes_dim6" / "circuit_config.json",
    },
    "nbS1-O1-E2Sst-maxNsyn-HEX0-L5": {
        "path": TINY / "nbS1-O1-E2Sst-maxNsyn-HEX0-L5" / "circuit_config.json",
    },
    # staging examples, only used when USE_STAGING is True:
    # "paired-neuron": {"id": "ed096991-4171-4cfc-b54e-756323fc62e7"},
    # "small-microcircuit": {"id": "4733054a-edc5-46cb-80cd-748758879355"},
}

N_PROCESSES = [
    int(x) for x in os.environ.get("BENCH_N_PROCESSES", "1,4,8").split(",")
]
_circuit_filter = [
    x for x in os.environ.get("BENCH_CIRCUITS", "").split(",") if x
]
if _circuit_filter:
    CIRCUITS = {k: v for k, v in CIRCUITS.items() if k in _circuit_filter}

MAX_FILTER_LOCATIONS = 100  # must match DEFAULT_MAX_FILTER_LOCATIONS
SKIP_IF_DONE = True

print(f"Circuits: {list(CIRCUITS)}")
print(f"n_processes values: {N_PROCESSES}")


## Resolve circuits and compile mechanisms

Local circuits are wrapped in `Circuit` objects; staging entries are staged via
`db_sdk.resolve_circuit` with `entity_cache=True`. MOD files are compiled once
per circuit (outside the timed region) via `nrnivmodl`.


In [ ]:
def ensure_mechanisms_compiled(circuit_dir: Path) -> None:
    """Run nrnivmodl on <circuit_dir>/mod if not already compiled."""
    mod_dir = circuit_dir / "mod"
    arch = "arm64" if platform.machine() == "arm64" else "x86_64"
    compiled_dir = circuit_dir / arch
    if not mod_dir.exists():
        print(f"  No mod/ directory at {mod_dir} - skipping compilation")
        return
    built = compiled_dir / "libnrnmech.dylib"
    if not built.exists():
        built = compiled_dir / "libnrnmech.so"
    if built.exists():
        print(f"  Mechanisms already compiled: {compiled_dir}")
        return
    print(f"  Compiling MOD files in {mod_dir}...")
    nrnivmodl = Path(sys.prefix) / "bin" / "nrnivmodl"
    if not nrnivmodl.exists():
        nrnivmodl = "nrnivmodl"
    result = subprocess.run(
        [str(nrnivmodl), "-incflags", "-DDISABLE_REPORTINGLIB", str(mod_dir)],
        cwd=circuit_dir,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"  nrnivmodl failed (exit {result.returncode}):")
        print(result.stderr[-2000:] if result.stderr else "(no stderr)")
    else:
        print(f"  MOD files compiled: {compiled_dir}")


def resolve_circuit_entry(name: str, spec: dict) -> Circuit:
    """Return a local Circuit object for one CIRCUITS entry."""
    if "path" in spec:
        path = Path(spec["path"]).resolve()
        if not path.exists():
            raise FileNotFoundError(f"Circuit config not found: {path}")
        return Circuit(name=name, path=str(path))
    if "id" in spec:
        if not USE_STAGING:
            raise ValueError(f"Circuit '{name}' needs USE_STAGING=True")
        ref = CircuitFromID(id_str=spec["id"])
        with tempfile.TemporaryDirectory() as tmp:
            circuit, _entity = db_sdk.resolve_circuit(
                ref,
                db_client=db_client,
                entity_cache=True,
                cache_root=output_root,
                temp_dir=Path(tmp),
            )
        return circuit
    raise ValueError(f"Circuit '{name}' has neither 'path' nor 'id'")


resolved_circuits = {}
for name, spec in CIRCUITS.items():
    print(f"Resolving {name}...")
    resolved_circuits[name] = resolve_circuit_entry(name, spec)
    ensure_mechanisms_compiled(Path(resolved_circuits[name].path).parent)


## Circuit metrics

In [ ]:
def circuit_metrics(circuit: Circuit) -> dict:
    """Node/synapse counts used by the resource estimator."""
    sonata = circuit.sonata_circuit
    n_bio = 0
    n_nodes_total = 0
    etypes = set()
    morphologies = set()
    for pop_name in sonata.nodes.population_names:
        pop = sonata.nodes[pop_name]
        n_nodes_total += pop.size
        if getattr(pop, "type", None) == "biophysical":
            n_bio += pop.size
            try:
                df = pop.get()
            except Exception:
                continue
            if "etype" in df.columns:
                etypes.update(df["etype"].dropna().unique())
            if "morphology" in df.columns:
                morphologies.update(df["morphology"].dropna().unique())
    n_synapses = 0
    for pop_name in sonata.edges.population_names:
        n_synapses += sonata.edges[pop_name].size
    return {
        "n_bio": int(n_bio),
        "n_nodes_total": int(n_nodes_total),
        "n_synapses": int(n_synapses),
        "n_etypes": len(etypes),
        "n_morphologies": len(morphologies),
    }


## run_one: a single benchmark run

Builds a `CircuitSimplificationScanConfig` for `single_compartment`, expands it
via `GridScanGenerationTask`, and times `CircuitSimplificationTask.execute()`
with `db_client=None` (no registration). The sampler wraps only the task
execution. Each run is appended to the CSV immediately so partial progress
survives a crash.


In [ ]:
CSV_PATH = output_root / "benchmarks" / "simplification_benchmarks.csv"


def append_result(row: dict) -> None:
    df_new = pd.DataFrame([row])
    if CSV_PATH.exists():
        df = pd.concat([pd.read_csv(CSV_PATH), df_new], ignore_index=True)
    else:
        CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
        df = df_new
    df.to_csv(CSV_PATH, index=False)


def run_one(
    circuit_name: str,
    circuit: Circuit,
    n_processes: int,
    output_root: Path,
) -> dict:
    run_dir = output_root / "benchmarks" / f"{circuit_name}__np{n_processes}"
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    row = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "hostname": socket.gethostname(),
        "machine": platform.machine(),
        "cpu_count": os.cpu_count(),
        "circuit_name": circuit_name,
        **circuit_metrics(circuit),
        "max_filter_locations": MAX_FILTER_LOCATIONS,
        "work_units": None,  # filled below
        "n_processes": n_processes,
        "algorithm": "single_compartment",
    }
    row["work_units"] = row["n_bio"] * MAX_FILTER_LOCATIONS

    sampler = PeakMemorySampler()
    try:
        scan_config = CircuitSimplificationScanConfig(
            info=Info(
                campaign_name=f"bench-{circuit_name}-np{n_processes}",
                campaign_description="Benchmark run for resource calibration",
            ),
            initialize=CircuitSimplificationScanConfig.Initialize(
                circuit=circuit,
                n_processes=n_processes,
            ),
            algorithms={"single_compartment": SingleCompartmentAlgorithm()},
        )
        grid_scan = obi.GridScanGenerationTask(
            form=scan_config,
            output_root=str(run_dir),
            coordinate_directory_option="ZERO_INDEX",
        )
        grid_scan.execute()
        with sampler:
            CircuitSimplificationTask(config=grid_scan.single_configs[0]).execute(
                db_client=None
            )
        row["status"] = "ok"
        row["error"] = ""
    except Exception as e:
        row["status"] = "error"
        row["error"] = f"{type(e).__name__}: {e}"
    row.update(sampler.results())
    append_result(row)
    return row


## Benchmark loop

In [ ]:
done = set()
if SKIP_IF_DONE and CSV_PATH.exists():
    prev = pd.read_csv(CSV_PATH)
    done = set(
        zip(
            prev.loc[prev.status == "ok", "circuit_name"],
            prev.loc[prev.status == "ok", "n_processes"],
        )
    )

for circuit_name, circuit in resolved_circuits.items():
    for n_proc in N_PROCESSES:
        if (circuit_name, n_proc) in done:
            print(f"SKIP {circuit_name} np={n_proc} (already in CSV)")
            continue
        print(f"RUN  {circuit_name} np={n_proc} ...", flush=True)
        row = run_one(circuit_name, circuit, n_proc, output_root)
        print(
            f"     -> {row['status']}: wall={row['wall_s']:.1f}s "
            f"peak_total={row['peak_rss_total_gb']:.2f}GB "
            f"peak_children={row['peak_rss_children_gb']:.2f}GB "
            f"{row['error']}"
        )


## Analysis

In [ ]:
df = pd.read_csv(CSV_PATH)
ok = df[df.status == "ok"].copy()
print(f"{len(ok)} successful runs, {len(df) - len(ok)} errors")
ok[
    [
        "circuit_name",
        "n_processes",
        "n_bio",
        "work_units",
        "wall_s",
        "peak_rss_main_gb",
        "peak_rss_children_gb",
        "peak_rss_total_gb",
    ]
]


### Time model -> `FILTER_WORK_UNITS_PER_CORE_HOUR`

For each run: `work_units_per_core_hour = work_units / (wall_s/3600 * n_processes)`.
Parallel efficiency = `wall_s(np=1) / (wall_s(np=k) * k)`.


In [ ]:
import matplotlib.pyplot as plt

ok["work_units_per_core_hour"] = ok["work_units"] / (
    ok["wall_s"] / 3600 * ok["n_processes"]
)
print(ok[["circuit_name", "n_processes", "wall_s", "work_units_per_core_hour"]])
print(
    f"\nMedian work_units_per_core_hour: "
    f"{ok['work_units_per_core_hour'].median():.0f}"
)

for circuit_name, grp in ok.groupby("circuit_name"):
    grp = grp.sort_values("n_processes")
    if 1 in set(grp.n_processes):
        t1 = grp.loc[grp.n_processes == 1, "wall_s"].iloc[0]
        for _, r in grp.iterrows():
            eff = t1 / (r.wall_s * r.n_processes)
            print(
                f"{circuit_name} np={r.n_processes}: "
                f"parallel efficiency {eff:.2f}"
            )

fig, ax = plt.subplots()
for n_proc, grp in ok.groupby("n_processes"):
    ax.plot(grp.work_units, grp.wall_s, "o-", label=f"np={n_proc}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("work_units (n_bio x max_filter_locations)")
ax.set_ylabel("wall_s")
ax.legend()
ax.set_title("Wall time vs work units")
plt.show()


### Memory model -> `BASE_MEMORY_GB` + `PER_WORKER_GB`

Per circuit, linear fit of `peak_rss_total_gb` vs `n_processes`:
slope = `PER_WORKER_GB`, intercept = `BASE_MEMORY_GB`.


In [ ]:
fits = {}
for circuit_name, grp in ok.groupby("circuit_name"):
    grp = grp.sort_values("n_processes")
    if len(grp) >= 2:
        slope, intercept = np.polyfit(grp.n_processes, grp.peak_rss_total_gb, 1)
        fits[circuit_name] = (intercept, slope)
        print(
            f"{circuit_name}: BASE_MEMORY_GB={intercept:.2f}, "
            f"PER_WORKER_GB={slope:.2f}"
        )
    else:
        print(f"{circuit_name}: need >=2 n_processes points for a fit")

fig, ax = plt.subplots()
for circuit_name, grp in ok.groupby("circuit_name"):
    grp = grp.sort_values("n_processes")
    ax.plot(grp.n_processes, grp.peak_rss_total_gb, "o-", label=circuit_name)
ax.set_xlabel("n_processes")
ax.set_ylabel("peak_rss_total_gb")
ax.legend()
ax.set_title("Peak total RSS vs n_processes")
plt.show()


### Estimator comparison: placeholder vs fitted constants

Reproduces the sizing logic of `estimate_task_resources` (work_units ->
ncpu_required -> mem_gb_required -> preset -> time_h) without touching
entitycore, and compares predictions against the measured runs.


In [ ]:
try:
    from app.services.resource_estimation.circuit_simplification import (
        BASE_MEMORY_GB,
        DEFAULT_MAX_FILTER_LOCATIONS,
        FILTER_WORK_UNITS_PER_CORE,
        FILTER_WORK_UNITS_PER_CORE_HOUR,
        PER_WORKER_GB,
        _get_required_cpu_memory_combo,
    )
except ModuleNotFoundError:
    # app/ is not an installed package - add the repo root to sys.path
    sys.path.insert(0, str(Path("../../../../../").resolve()))
    from app.services.resource_estimation.circuit_simplification import (
        BASE_MEMORY_GB,
        DEFAULT_MAX_FILTER_LOCATIONS,
        FILTER_WORK_UNITS_PER_CORE,
        FILTER_WORK_UNITS_PER_CORE_HOUR,
        PER_WORKER_GB,
        _get_required_cpu_memory_combo,
    )


def predict(n_bio, *, filter_wupc, wupch, per_worker_gb, base_gb):
    work_units = n_bio * DEFAULT_MAX_FILTER_LOCATIONS
    ncpu_required = max(1, int(np.ceil(work_units / filter_wupc)))
    ncpu_required = min(ncpu_required, 16)  # launch-system max
    mem_gb_required = base_gb + ncpu_required * per_worker_gb
    ncpu, mem_gb = _get_required_cpu_memory_combo(ncpu_required, mem_gb_required)
    time_h = max(1, int(np.ceil(work_units / (ncpu * wupch))))
    return {
        "work_units": work_units,
        "ncpu": ncpu,
        "mem_gb": mem_gb,
        "time_h": time_h,
    }


scenarios = {
    "placeholder": {
        "filter_wupc": FILTER_WORK_UNITS_PER_CORE,
        "wupch": FILTER_WORK_UNITS_PER_CORE_HOUR,
        "per_worker_gb": PER_WORKER_GB,
        "base_gb": BASE_MEMORY_GB,
    },
}
if fits:
    fit_intercept = float(np.median([v[0] for v in fits.values()]))
    fit_slope = float(np.median([v[1] for v in fits.values()]))
    scenarios["fitted"] = {
        "filter_wupc": FILTER_WORK_UNITS_PER_CORE,
        "wupch": float(ok["work_units_per_core_hour"].median()),
        "per_worker_gb": fit_slope,
        "base_gb": fit_intercept,
    }

rows = []
for _, r in ok.iterrows():
    for scen_name, params in scenarios.items():
        p = predict(r.n_bio, **params)
        rows.append(
            {
                "scenario": scen_name,
                "circuit_name": r.circuit_name,
                "n_processes": r.n_processes,
                "pred_ncpu": p["ncpu"],
                "pred_mem_gb": p["mem_gb"],
                "pred_time_h": p["time_h"],
                "meas_wall_s": r.wall_s,
                "meas_peak_total_gb": r.peak_rss_total_gb,
                "would_fit": bool(
                    r.peak_rss_total_gb <= p["mem_gb"]
                    and r.wall_s <= p["time_h"] * 3600
                ),
            }
        )
pd.DataFrame(rows)


## Next steps

- Update `FILTER_WORK_UNITS_PER_CORE`, `FILTER_WORK_UNITS_PER_CORE_HOUR`,
  `BASE_MEMORY_GB`, and `PER_WORKER_GB` in
  `app/services/resource_estimation/circuit_simplification.py` from the fitted
  values above (with headroom - these are peaks, not averages).
- Re-check the numbers on the launch system (different CPU/memory profile than
  a laptop).
- Extend the benchmark to the point-neuron algorithms (`lif`, `adex`,
  `izhikevich`, `glif`, `gif`) to calibrate `POINT_NEURON_WORK_UNITS_PER_CORE*`.
